# Jumper Notebook: PCA + GP Forecast Workflow

This notebook follows the refactored helper-based pipeline:
1. setup and parameters
2. load and prepare simulations
3. PCA decomposition and diagnostics
4. component-space forecasting
5. reconstruction and save
6. error evaluation
7. component-space comparison against a reference window


In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from nemo_spinup_forecast.dimensionality_reduction import (
    dimensionality_reduction_techniques,
)
from nemo_spinup_forecast.forecast_method import forecast_techniques
from nemo_spinup_forecast.pipeline_utils import (
    TermSpec,
    abs_error_stats,
    build_predictions,
    build_simulations,
    compute_rmse_for_terms,
    decompose_all,
    forecast_all,
    load_ts_all,
    make_dicos,
    normalise_time_series,
)
from nemo_spinup_forecast.plotting_utils import (
    plot_bar_with_errors,
    plot_component_timeseries,
    plot_depth_error_profiles,
    plot_depth_prediction_reference,
    plot_pca_diagnostics,
    plot_reconstructions,
    plot_rmse_depth_profile,
    plot_rmse_maps,
    plot_simulation_snapshots,
)
from nemo_spinup_forecast.utils import (
    create_run_dir,
    load_ocean_terms,
)

## Parameters and Outputs

### Inputs and slicing
- `data_path`: root directory containing DINO simulation files.
- `ye`: yearly processing flag passed to `Simulation`.
- `start`, `end`: slice bounds for the training/reference window.
- `comp_value`: number of components used, or amount of variance kept, passed to dimensionality reduction.

### Forecast horizon
- `steps`: number of forecast time steps.

### Run-scoped outputs
- `run_dir`: timestamped path created by `create_run_dir(data_path)`.
- `out_dir`: `<run_dir>/forecasts`.
- `prepared_dir`: `<out_dir>/simu_prepared` — one subdirectory per term (e.g. `ssh/ssh.npz`, `ssh/pca_ssh`).
- `pred_dir`: `<out_dir>/simu_predicted` for reconstructed fields (`pred_*.npy`).

In [ ]:
data_path = str(Path("../tests/data/nemo_data_e3/").resolve())
run_dir = create_run_dir(str(data_path))
out_dir = Path(run_dir)
prepared_dir = out_dir / "simu_prepared"
pred_dir = out_dir / "simu_predicted"

ye = True

start = 10
end = 40
comp_value = 3  # number of components used, or amount of variance kept
steps = 10

## 1) Load and prepare simulations


This section resolves ocean-term names, defines `TERM_SPECS`, and calls
`build_simulations(...)`.

`build_simulations(...)` loads the `start:end` slice and standardizes it with
`(value - mean) / (2 * std)` before PCA.

`TERM_SPECS` also defines later reductions:
- `mean_axes` for mean profiles.
- `err_axes` for error summaries.

SSH (`zos`) has shape `(time, lat, lon)`, so reducing space gives a time series.
Salinity and temperature have shape `(time, depth, lat, lon)`, so reducing time and
horizontal axes gives a depth profile.


In [ ]:
# Load ocean term definitions from packaged config.
term_defs = load_ocean_terms()
display_names = {td.key: td.term for td in term_defs}

# Technique selection — change the dict key to swap technique
dimensionality_reduction_method = dimensionality_reduction_techniques["PCA"]
forecast_method = forecast_techniques["GaussianProcessForecaster"]

# `mean_axes` and `err_axes` depend on the field shape.
# SSH is stored as (time, lat, lon), so collapsing (lat, lon) leaves a time series.
# Salinity and temperature are stored as (time, depth, lat, lon), so collapsing
# (time, lat, lon) leaves a depth profile.
AXES_CONFIG = {
    "ssh": {"mean_axes": (1, 2), "err_axes": (0, 1, 2)},
    "salinity": {"mean_axes": (0, 2, 3), "err_axes": (0, 2, 3)},
    "temperature": {"mean_axes": (0, 2, 3), "err_axes": (0, 2, 3)},
}

TERM_SPECS = [
    TermSpec(
        key=td.key,
        term=td.term,
        filename=td.filename,
        **AXES_CONFIG[td.key],
    )
    for td in term_defs
]


sims = build_simulations(
    TERM_SPECS,
    data_path=data_path,
    start=start,
    end=end,
    comp=comp_value,
    ye=ye,
    dr_method=dimensionality_reduction_method,
    stand=True,
)

In [ ]:
# Snapshot plots use standardized training data.
# Salinity edge values near -1.75 come from raw 0.0 boundary cells after scaling.
# They are not negative physical salinity.
simus = [sims["ssh"], sims["salinity"], sims["temperature"]]
# Matching display names for plots.
names = [display_names["ssh"], display_names["salinity"], display_names["temperature"]]
print(
    sims["salinity"].simulation[0, 0],
    sims["salinity"].simulation.shape,
    sims["salinity"].simulation[0, 0].shape,
)
plot_simulation_snapshots(simus, names);

## 2) PCA decomposition and diagnostics


- `decompose_all(sims)` applies dimensionality reduction for each term.
- `compute_rmse_for_terms(...)` computes reconstruction RMSE with all fitted components.
- `rmseVs["zos"]` has shape `(time,)`: one RMSE value per SSH field and time step.
- `rmseVs["so"]` and `rmseVs["thetao"]` have shape `(time, depth)`: one RMSE value per
  depth slice and time step.
- `rmseMs["zos"]` has shape `(lat, lon)`: RMSE at each SSH grid point over time.
- `rmseMs["so"]` and `rmseMs["thetao"]` have shape `(depth, lat, lon)`: the map plot
  averages error over depth.
- `make_dicos(sims)` builds the saved `*.npz` and `pca_*` outputs.


In [ ]:
# Apply dimensionality reduction to each simulation.
decompose_all(sims)

In [ ]:
colors = ["tab:blue", "tab:green", "tab:red"]

plot_pca_diagnostics(simus, names, colors);

In [ ]:
recs, rmseVs, rmseMs = compute_rmse_for_terms(TERM_SPECS, sims)

In [ ]:
import xarray as xr

# Load simulation data to get total depth levels
array = xr.open_dataset(
    sims["salinity"].files[-1], decode_times=False, chunks={"time": 200, "x": 120}
)

In [ ]:
maps = [rmseMs["ssh"], rmseMs["salinity"], rmseMs["temperature"]]
names = [display_names["ssh"], display_names["salinity"], display_names["temperature"]]
colors = ["tab:blue", "darkgreen", "darkred"]
# Only 4D variables (salinity, temperature) have a depth axis; SSH is excluded.
plot_rmse_depth_profile(
    [rmseVs["salinity"].T, rmseVs["temperature"].T],
    array.deptht,
    [display_names["salinity"], display_names["temperature"]],
    ["darkgreen", "darkred"],
    "PCA reconstruction RMSE by depth",
);

In [ ]:
# SSH is plotted directly as a 2D RMSE map.
# Salinity and temperature RMSE maps are averaged over depth first.
plot_rmse_maps(maps, names);

In [ ]:
dicos = make_dicos(sims)

In [ ]:
# TERM_SPECS already stores both the internal key and the file term.
# Use spec.key for dict lookups and spec.term for saved filenames.
# Each term gets its own subdirectory, matching Simulation.save() convention.
for spec in TERM_SPECS:
    term_dir = prepared_dir / spec.term
    term_dir.mkdir(parents=True, exist_ok=True)
    with open(term_dir / f"pca_{spec.term}", "wb") as file:
        pickle.dump(sims[spec.key].pca, file)
    np.savez(term_dir / spec.term, **dicos[spec.key])

## 3) Forecast in component space


- `load_ts_all(prepared_path, TERM_SPECS)` loads prepared component time series.
- `build_predictions(...)` creates one `Predictions` object per term.
- `Predictions.forecast_single_series(...)` is used as a single-component demo.
- No notebook seed is set here. The default GP regressor already uses a fixed
  `random_state=42`.
- `forecast_all(...)` returns raw forecast component tables keyed by term.
- The notebook prepends the training window before reconstruction.
- `Simulation.reconstruct(...)` maps predicted components back to physical space.

In [ ]:
prepared_path = str(prepared_dir)


# Load one PCA component time-series table and one metadata dict per term.
component_ts_by_term, metadata_by_term = load_ts_all(prepared_path, TERM_SPECS)


preds = build_predictions(
    TERM_SPECS,
    component_ts_by_term,
    metadata_by_term,
    forecast_method,
    dimensionality_reduction_method,
)

In [ ]:
comp, train_len = 1, len(preds["ssh"])
colors = {"ssh": None, "salinity": "darkgreen", "temperature": "darkred"}

# Single-component forecast for component `comp`.
single_forecast_by_term, single_forecast_std_by_term, single_forecast_metrics_by_term = (
    {},
    {},
    {},
)
for spec in TERM_SPECS:
    hat, std, metrics = preds[spec.key].forecast_single_series(comp, train_len, steps)
    single_forecast_by_term[spec.key] = hat
    single_forecast_std_by_term[spec.key] = std
    single_forecast_metrics_by_term[spec.key] = metrics

print("single_forecast_ssh shape:", single_forecast_by_term["ssh"].shape)

# Reuse Predictions.show for quick visual inspection.
for spec in TERM_SPECS:
    c = colors[spec.key]
    if c is None:
        preds[spec.key].show(
            comp,
            single_forecast_by_term[spec.key],
            single_forecast_std_by_term[spec.key],
            train_len,
        )
    else:
        preds[spec.key].show(
            comp,
            single_forecast_by_term[spec.key],
            single_forecast_std_by_term[spec.key],
            train_len,
            color=c,
        )

In [ ]:
(
    forecast_component_ts_by_term,
    forecast_component_std_by_term,
    forecast_metrics_by_term,
) = forecast_all(TERM_SPECS, preds, train_len=train_len, steps=steps)
# `forecast_all(...)` returns forecast rows only.
# Prepend the training window before reconstruction.
forecast_component_ts_by_term = {
    k: pd.concat([component_ts_by_term[k][:train_len], h])
    for k, h in forecast_component_ts_by_term.items()
}

In [ ]:
# Reconstruct physical-space fields from predicted component series.
predictions = {}
for spec in TERM_SPECS:
    n = np.shape(preds[spec.key].info["ts"])[1]
    predictions[spec.key] = sims[spec.key].reconstruct(
        forecast_component_ts_by_term[spec.key], n, preds[spec.key].info
    )
    print(f"{spec.key} reconstructed with all comp")

In [ ]:
maps = [predictions["ssh"], predictions["salinity"], predictions["temperature"]]
names = [display_names["ssh"], display_names["salinity"], display_names["temperature"]]

# Plot one reconstructed layer for each ocean field.
plot_reconstructions(maps, names);

## 4) Save reconstructed predictions


In [ ]:
# Save reconstructed predicted fields.
pred_dir.mkdir(exist_ok=True)

# Use the external file term from TERM_SPECS when saving outputs.
for spec in TERM_SPECS:
    np.save(pred_dir / f"pred_{spec.term}.npy", predictions[spec.key])

## 5) Evaluate prediction errors in physical space


In [ ]:
# Reload predictions using the same term metadata as the save step.
pred_arrays = {
    spec.key: np.load(pred_dir / f"pred_{spec.term}.npy") for spec in TERM_SPECS
}

In [ ]:
id_, start2, end2 = "106", start, end + steps  # start, end + steps
ye = True
# Build reference simulations over training + forecast window.
ref = build_simulations(
    TERM_SPECS,
    data_path=data_path,
    start=start2,
    end=end2,
    comp=comp_value,
    ye=ye,
    dr_method=dimensionality_reduction_method,
    # Keep reference arrays on original scale for direct comparison.
    stand=False,
)

In [ ]:
# Load simulation data to read depth levels for profile plots.
array = xr.open_dataset(
    ref["salinity"].files[-1], decode_times=False, chunks={"time": 200, "x": 120}
)
depth = array.deptht
del array

### Absolute error summary (forecast vs pre-forecast)

`abs_error_stats(...)` splits each error array into two windows:
- `pred_*`: the last `steps` samples, i.e. the forecast window.
- `ref_*`: the earlier training window.

SSH is reduced to one mean absolute error per window.
Salinity and temperature keep depth, so the profiles show mean absolute error by depth.

The last plots compare mean predicted and reference profiles.


In [ ]:
# Absolute errors between reconstructed predictions and references.
# pred_arrays are loaded in the previous cell.
print(ref)
errs = {k: np.abs(ref[k].simulation - pred_arrays[k]) for k in pred_arrays}

print("pred_ssh shape:", pred_arrays["ssh"].shape)
print("ref_ssh shape:", ref["ssh"].simulation.shape)
# Split errors into forecast window and pre-forecast baseline window.
error_stats = {}
for spec in TERM_SPECS:
    error_stats[spec.key] = abs_error_stats(
        errs[spec.key],
        steps=steps,
        axes=spec.err_axes,
    )

# Display summary of aggregated error-statistic shapes.
summary_rows = []
for spec in TERM_SPECS:
    s = error_stats[spec.key]
    summary_rows.append(
        {
            "term": spec.key,
            "pred_mean_shape": np.shape(s["pred_mean"]),
            "ref_mean_shape": np.shape(s["ref_mean"]),
        }
    )
pd.DataFrame(summary_rows)

In [ ]:
categories = ["Forecast", "Pre-forecast"]

# Plot SSH mean absolute error with standard-deviation bars.
means = [error_stats["ssh"]["pred_mean"], error_stats["ssh"]["ref_mean"]]
print(means)
errors = [error_stats["ssh"]["pred_std"], error_stats["ssh"]["ref_std"]]

plot_bar_with_errors(
    categories,
    means,
    errors,
    "Mean Error with Standard Error",
    "Error",
);

In [ ]:
# Plot depth-wise mean absolute errors for temperature and salinity.
plot_depth_error_profiles(
    depth,
    [error_stats["temperature"]["ref_mean"], error_stats["salinity"]["ref_mean"]],
    [error_stats["temperature"]["ref_std"], error_stats["salinity"]["ref_std"]],
    [error_stats["temperature"]["pred_mean"], error_stats["salinity"]["pred_mean"]],
    [error_stats["temperature"]["pred_std"], error_stats["salinity"]["pred_std"]],
    ["temperature", "salinity"],
    ["darkred", "darkgreen"],
    f"Absolute error on {steps} last predictions",
);

In [ ]:
# Mean profiles using axes declared in TERM_SPECS.
# salinity/temperature: mean over (time, lat, lon) -> depth profile.
# ssh: mean over (lat, lon) -> time series.


mean_pred, mean_ref = {}, {}
for spec in TERM_SPECS:
    mean_pred[spec.key] = np.nanmean(pred_arrays[spec.key], axis=spec.mean_axes)
    mean_ref[spec.key] = np.nanmean(ref[spec.key].simulation, axis=spec.mean_axes)

print({k: v.shape for k, v in mean_pred.items()})
print({k: v.shape for k, v in mean_ref.items()})

In [ ]:
print("depth shape: ", depth.shape)
print("mean_pred_salinity shape: ", mean_pred["salinity"].shape)

# Compare mean predicted and reference profiles.
plot_depth_prediction_reference(
    depth,
    [mean_pred["salinity"], mean_pred["temperature"]],
    [mean_ref["salinity"], mean_ref["temperature"]],
    ["Salinity", "Temperature"],
);

## 6) Compare forecasted component time series with separately fitted PCA reference component

The forecasted component series come from the original training PCA fit.
The grey reference lines come from a fresh PCA fit on the longer reference window.
These plots can be used as a qualitative view, not as a direct component-by-component check.


In [ ]:
# Normalize the longer reference window and do a PCA decomposition.

for spec in TERM_SPECS:
    normalise_time_series(ref[spec.key])


decompose_all(ref)

In [ ]:
simus = [ref["ssh"], ref["salinity"], ref["temperature"]]
names = [display_names["ssh"], display_names["salinity"], display_names["temperature"]]
colors = ["tab:blue", "tab:green", "tab:red"]

plot_pca_diagnostics(simus, names, colors);

In [ ]:
comp = 0
ref_list = [ref["ssh"], ref["salinity"], ref["temperature"]]
pred = [
    forecast_component_ts_by_term["ssh"],
    forecast_component_ts_by_term["salinity"],
    forecast_component_ts_by_term["temperature"],
]
names = [display_names["ssh"], display_names["salinity"], display_names["temperature"]]
colors = ["tab:blue", "tab:green", "tab:red"]
total_len = len(ref_list[0].simulation)

plot_component_timeseries(ref_list, pred, names, colors, comp, total_len, train_len);

In [ ]:
comp = 1

plot_component_timeseries(ref_list, pred, names, colors, comp, total_len, train_len);

In [ ]:
comp = 2

plot_component_timeseries(ref_list, pred, names, colors, comp, total_len, train_len);